# MIND Large — Real Codabench Test Submission

Generates the actual Codabench submission for MIND (`codabench.org/competitions/13967`)
against `MINDlarge_test` -- the real, blind held-out population, downloaded
separately as `MINDlarge_test.zip` (not part of `data/raw`'s tracked datasets,
gitignored). Unlike every other track in this repo, this file's `behaviors.tsv`
carries **no click labels at all** (`impressions` is a plain space-separated
candidate list, no `-0`/`-1` suffix) -- it is a genuine blind test, not a
held-out split of already-labeled data. This is why it is a standalone
notebook rather than a sixth track through `build_pipeline.ipynb`/`bm25_retrieval.ipynb`/
`embedding_retrieval.ipynb`/`evaluation_harness.ipynb`: Q2/Q3's recall@K and
Q4's ranking metrics all require ground-truth clicks that don't exist here.
Only Q5's re-ranking task applies.

`MINDlarge_test`'s 120,961-article catalog overlaps heavily but not fully
with `mind_large`'s existing (train/dev-derived) catalog -- 26,228 articles
appear only here. Those need a fresh Kaggle embeddings pass; the other
94,733 reuse `mind_large`'s already-computed embeddings directly (the
embedding model is frozen/pretrained -- computing an embedding for an
article's text is a deterministic, label-free transform, not something
that can leak test-set click information; see PROMPTS.md for the
discussion this came out of).

Run top-to-bottom in two sittings (this notebook pauses for the Kaggle
step -- see the markdown cell marked **PAUSE HERE**) to rebuild
`submissions/mind_large_test/mind_large_test_{method}_predictions.zip`.

## Setup

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import zipfile

import numpy as np
import polars as pl

from cs4406m26_assignment1c1.bm25 import tokenize, build_index, get_scores
from cs4406m26_assignment1c1.embeddings import mean_pool, cosine_similarity_subset, normalize_rows


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
RAW_DIR = ROOT / "MINDlarge_test"
OUT_DIR = ROOT / "data" / "processed" / "mind_large_test"
SUBMISSIONS_DIR = ROOT / "submissions" / "mind_large_test"
PROGRESS_LOG = ROOT / "build_progress.log"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

PREFIX = "mind_large_test_"
RECENT_N_CLICKS = 20
METHODS = ["embedding", "bm25"]


def log_progress(message: str) -> None:
    with PROGRESS_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{datetime.now(timezone.utc).isoformat()}] {message}\n")
        f.flush()


if not (RAW_DIR / "news.tsv").exists():
    raise FileNotFoundError(
        f"missing {RAW_DIR / 'news.tsv'} -- unzip MINDlarge_test.zip at the repo root first "
        "(it must extract to ./MINDlarge_test/)."
    )

log_progress("mind_large_test_submission started")
{"raw_dir": str(RAW_DIR), "out_dir": str(OUT_DIR), "submissions_dir": str(SUBMISSIONS_DIR)}

{'raw_dir': 'C:\\Users\\HP\\cs4406m26-assignment1c1\\MINDlarge_test',
 'out_dir': 'C:\\Users\\HP\\cs4406m26-assignment1c1\\data\\processed\\mind_large_test',
 'submissions_dir': 'C:\\Users\\HP\\cs4406m26-assignment1c1\\submissions\\mind_large_test'}

## Parse articles (`news.tsv`)

Same `polars` + `quote_char=None` approach as `build_pipeline.ipynb`'s MIND
parsing (SPEC.md Q1 #5) -- `news.tsv` is a raw TSV with no CSV-style
escaping convention, and pandas' default quote handling silently corrupts
titles/abstracts containing a literal `"`. Matches `mind_large`'s unified
`articles` schema exactly (`dataset`, `body`/`published_time` null-filled --
MIND never has either, same as every other MIND-family dataset) so
`data/processed/mind_large_test/` has the same shape as every other dataset
directory. The one column this notebook's tables can *never* carry --
`article_ids_clicked`, in `behaviors` below -- is a structural fact about
this being a genuine blind test, not a MIND-format limitation like the
null columns here, so it's left out entirely rather than null-filled (see
the intro markdown).

In [2]:
NEWS_COLUMNS = ["news_id", "category", "subcategory", "title", "abstract", "url", "title_entities", "abstract_entities"]

news_raw = pl.read_csv(
    RAW_DIR / "news.tsv", separator="\t", has_header=False, new_columns=NEWS_COLUMNS, quote_char=None,
)

articles = news_raw.select(
    (pl.lit(PREFIX) + pl.col("news_id")).alias("article_id"),
    pl.lit("mind_large_test").alias("dataset"),
    pl.col("title"),
    pl.col("abstract"),
    pl.lit(None, dtype=pl.Utf8).alias("body"),  # MIND never has full body text -- same as every other MIND dataset
    pl.col("category"),
    pl.col("subcategory"),
    pl.lit(None, dtype=pl.Utf8).alias("published_time"),  # MIND's news.tsv carries no publish date at all
)
log_progress(f"mind_large_test: parsed {articles.height} articles from news.tsv")
articles.write_parquet(OUT_DIR / "articles.parquet")
articles.shape

(120961, 8)

In [3]:
def test_articles_parsed():
    assert articles.height == news_raw.height
    assert articles["article_id"].n_unique() == articles.height, "duplicate article_id"
    assert articles["article_id"].str.starts_with(PREFIX).all()
    assert (articles["dataset"] == "mind_large_test").all()
    assert articles["body"].is_null().all() and articles["published_time"].is_null().all()
    assert articles["category"].null_count() == 0

    # schema parity with mind_large's own unified articles table (see SPEC.md Q1 #2)
    existing_schema = pl.read_parquet_schema(ROOT / "data" / "processed" / "mind_large" / "articles.parquet")
    assert set(articles.columns) == set(existing_schema.keys())

    # sanity floor -- MINDlarge_test's real catalog is on the order of 100K+ articles
    assert articles.height > 50_000


test_articles_parsed()
print(f"ok: {articles.height} articles parsed, unique, correctly prefixed, schema matches mind_large's articles table")

ok: 120961 articles parsed, unique, correctly prefixed, schema matches mind_large's articles table


## Identify articles needing fresh embeddings

`mind_large`'s existing `article_embeddings.parquet` already covers whichever
of this catalog's articles overlap with `MINDlarge_train`/`MINDlarge_dev` --
those are reused as-is (see the intro markdown for why re-encoding them
would be wasteful, not just slower). Only articles that exist *exclusively*
in `MINDlarge_test`'s own catalog need a Kaggle pass.

In [4]:
existing_mind_large_articles = pl.read_parquet(
    ROOT / "data" / "processed" / "mind_large" / "articles.parquet", columns=["article_id"]
)
existing_raw_ids = set(a.removeprefix("mind_large_") for a in existing_mind_large_articles["article_id"].to_list())
test_raw_ids = set(news_raw["news_id"].to_list())
missing_raw_ids = test_raw_ids - existing_raw_ids

# Kaggle upload only needs article_id/title/abstract -- compute_embeddings_kaggle.ipynb
# doesn't use the other unified-schema columns, and this file is a transient
# upload artifact, not one of the persisted data/processed/ tables.
new_articles_for_kaggle = articles.filter(
    pl.col("article_id").is_in([PREFIX + rid for rid in missing_raw_ids])
).select("article_id", "title", "abstract")
kaggle_upload_path = OUT_DIR / "mind_large_test_new_articles.parquet"
new_articles_for_kaggle.write_parquet(kaggle_upload_path)
log_progress(
    f"mind_large_test: {len(test_raw_ids)} total articles, {len(missing_raw_ids)} need fresh embeddings "
    f"-- wrote {kaggle_upload_path}"
)
{"total_articles": len(test_raw_ids), "already_have_embeddings": len(test_raw_ids) - len(missing_raw_ids),
 "need_kaggle_embeddings": len(missing_raw_ids)}

{'total_articles': 120961,
 'already_have_embeddings': 94733,
 'need_kaggle_embeddings': 26228}

In [5]:
def test_new_articles_for_kaggle():
    assert new_articles_for_kaggle.height == len(missing_raw_ids)
    assert new_articles_for_kaggle["article_id"].str.starts_with(PREFIX).all()
    assert set(a.removeprefix(PREFIX) for a in new_articles_for_kaggle["article_id"].to_list()) == missing_raw_ids
    # every missing id must actually be absent from the existing catalog, and
    # every non-missing id must actually be present -- catches an inverted filter
    assert missing_raw_ids.isdisjoint(existing_raw_ids)
    assert (test_raw_ids - missing_raw_ids).issubset(existing_raw_ids)
    assert kaggle_upload_path.exists()


test_new_articles_for_kaggle()
print(f"ok: {new_articles_for_kaggle.height} new articles written for Kaggle embedding")

ok: 26228 new articles written for Kaggle embedding


## PAUSE HERE — run `compute_embeddings_kaggle.ipynb` on Kaggle

1. Upload `data/processed/mind_large_test/mind_large_test_new_articles.parquet`
   as a Kaggle Dataset (Kaggle → New Notebook → **+ Add Data → Upload → New
   Dataset**) — already named correctly for the notebook's discovery pattern,
   no local rename needed.
2. Import [`src/compute_embeddings_kaggle.ipynb`](src/compute_embeddings_kaggle.ipynb).
3. Settings: **Accelerator → GPU T4**, **Internet → On**.
4. **Save Version → Save & Run All (Commit)**.
5. From that version's **Output** tab, download
   `mind_large_test_new_article_embeddings.parquet`.
6. Place it at `data/processed/mind_large_test/new_article_embeddings.parquet`.
7. Resume this notebook from the next cell.

## Resume: merge embeddings

Both sources are keyed by `article_id` under their *own* prefix (`mind_large_`
for the existing catalog, `mind_large_test_` for the freshly Kaggle-computed
subset) — re-key both to the bare MIND `news_id`, merge, then re-prefix
uniformly with `PREFIX` so the combined table matches `articles`'s own id
space exactly.

In [6]:
new_embeddings_path = OUT_DIR / "new_article_embeddings.parquet"
if not new_embeddings_path.exists():
    raise FileNotFoundError(
        f"missing {new_embeddings_path}. Run compute_embeddings_kaggle.ipynb on Kaggle for "
        "mind_large_test_new_articles.parquet first (see the PAUSE HERE cell above)."
    )

existing_embeddings_raw = pl.read_parquet(ROOT / "data" / "processed" / "mind_large" / "article_embeddings.parquet")
new_embeddings_raw = pl.read_parquet(new_embeddings_path)

existing_lookup = {
    aid.removeprefix("mind_large_"): emb
    for aid, emb in zip(existing_embeddings_raw["article_id"].to_list(), existing_embeddings_raw["embedding"].to_list())
}
new_lookup = {
    aid.removeprefix(PREFIX): emb
    for aid, emb in zip(new_embeddings_raw["article_id"].to_list(), new_embeddings_raw["embedding"].to_list())
}
combined_lookup = {**existing_lookup, **new_lookup}

missing_after_merge = test_raw_ids - set(combined_lookup)
if missing_after_merge:
    raise ValueError(f"{len(missing_after_merge)} test articles still have no embedding after merge")

embeddings_df = pl.DataFrame({
    "article_id": [PREFIX + rid for rid in test_raw_ids],
    "dataset": "mind_large_test",
    "embedding": [combined_lookup[rid] for rid in test_raw_ids],
})
embeddings_df.write_parquet(OUT_DIR / "article_embeddings.parquet")

n_reused = len(test_raw_ids) - len(missing_raw_ids)
n_fresh = len(missing_raw_ids)
log_progress(
    f"mind_large_test: merged embeddings for all {len(test_raw_ids)} articles "
    f"({n_reused} reused, {n_fresh} freshly computed)"
)
{"total": len(test_raw_ids), "reused": n_reused, "fresh": n_fresh}

{'total': 120961, 'reused': 94733, 'fresh': 26228}

In [7]:
def test_merged_embeddings():
    reloaded = pl.read_parquet(OUT_DIR / "article_embeddings.parquet")
    assert reloaded.height == articles.height
    assert set(reloaded["article_id"].to_list()) == set(articles["article_id"].to_list())
    assert len(reloaded["embedding"][0]) == 768

    # spot-check: an overlapping article's embedding is reused byte-identical
    # from mind_large, not recomputed -- catches a bug where the merge
    # silently re-encodes everything instead of actually reusing anything.
    overlap_raw_id = next(iter(test_raw_ids - missing_raw_ids))
    reused_vec = reloaded.filter(pl.col("article_id") == PREFIX + overlap_raw_id)["embedding"][0]
    original_vec = existing_embeddings_raw.filter(pl.col("article_id") == "mind_large_" + overlap_raw_id)["embedding"][0]
    assert list(reused_vec) == list(original_vec), "overlapping article's embedding should be reused exactly"


test_merged_embeddings()
print("ok: article_embeddings.parquet covers all test articles, overlapping embeddings reused exactly")

ok: article_embeddings.parquet covers all test articles, overlapping embeddings reused exactly


## Build BM25 index and embedding corpus matrix

Same construction as Q4/Q5 (`id_to_idx`/`corpus_unit` precomputed once here,
not per impression -- see SPEC.md Q4 #9 for why that matters at scale).

In [8]:
texts = (articles["title"].fill_null("") + " " + articles["abstract"].fill_null("")).to_list()
doc_tokens = [tokenize(t) for t in texts]
bm25_index = build_index(articles["article_id"].to_list(), doc_tokens)
title_by_id = dict(zip(articles["article_id"].to_list(), articles["title"].to_list()))

doc_ids = articles["article_id"].to_numpy()
emb_lookup = dict(zip(
    embeddings_df["article_id"].to_list(),
    [np.asarray(v) for v in embeddings_df["embedding"].to_list()],
))
matrix = np.stack([emb_lookup[aid] for aid in doc_ids]).astype(np.float32)
id_to_idx = {aid: i for i, aid in enumerate(doc_ids)}
corpus_unit = normalize_rows(matrix)

log_progress(f"mind_large_test: BM25 index ({bm25_index.n_docs} docs) and embedding matrix {matrix.shape} built")
{"bm25_docs": bm25_index.n_docs, "avgdl": bm25_index.avgdl, "embedding_matrix_shape": matrix.shape}

{'bm25_docs': 120961,
 'avgdl': np.float64(48.66098163871),
 'embedding_matrix_shape': (120961, 768)}

In [9]:
def test_bm25_and_embedding_corpus():
    assert bm25_index.n_docs == articles.height
    assert bm25_index.avgdl > 0
    assert matrix.shape == (articles.height, 768)
    assert not np.isnan(matrix).any()
    assert (doc_ids == articles["article_id"].to_numpy()).all()


test_bm25_and_embedding_corpus()
print("ok: BM25 index and embedding matrix built over the full test-article corpus, aligned, no NaNs")

ok: BM25 index and embedding matrix built over the full test-article corpus, aligned, no NaNs


## Parse behaviors (`behaviors.tsv`)

`impressions` here is a **plain space-separated candidate list, no
`-label` suffix** -- the real distinguishing feature of a blind test file
(see the intro markdown). `impression_id` (column 1) is already the correct
native ID for the submission format -- no dataset-prefix stripping needed,
since this notebook never namespaces impression IDs the way the unified
multi-dataset feature store does.

Matches `mind_large`'s unified `behaviors`/`history` schemas otherwise:
`session_id`/the three `history` sequence columns are null-filled (MIND
never has them, same as every other MIND dataset); `split` is set to the
constant `"test"` since this entire file *is* the held-out population, not
something this notebook derives via a temporal cutoff. The one column
genuinely absent -- `article_ids_clicked` -- stays absent rather than
null-filled; see the intro markdown for why that one is different in kind
from the others.

In [10]:
BEHAVIORS_COLUMNS = ["impression_id", "user_id", "time", "history", "impressions"]

behaviors_raw = pl.read_csv(
    RAW_DIR / "behaviors.tsv", separator="\t", has_header=False, new_columns=BEHAVIORS_COLUMNS,
    quote_char=None, schema_overrides={"impression_id": pl.Utf8},
)


def split_ids(col_name: str) -> pl.Expr:
    return (
        pl.col(col_name).fill_null("")
        .str.split(" ")
        .list.eval(pl.element().filter(pl.element() != ""))
        .list.eval(pl.lit(PREFIX) + pl.element())
    )


behaviors = behaviors_raw.select(
    pl.col("impression_id"),
    pl.lit("mind_large_test").alias("dataset"),
    (pl.lit(PREFIX) + pl.col("user_id")).alias("user_id"),
    pl.col("time").str.strptime(pl.Datetime, "%m/%d/%Y %I:%M:%S %p").alias("impression_time"),
    split_ids("history").alias("history_ids"),
    split_ids("impressions").alias("article_ids_inview"),
    pl.lit(None, dtype=pl.Utf8).alias("session_id"),  # MIND never has this -- same as every other MIND dataset
    pl.lit("test").alias("split"),  # this whole file *is* the held-out population, not a derived cutoff
)

history_table = (
    behaviors.select("user_id", "history_ids")
    .unique(subset=["user_id"], keep="first")
    .rename({"history_ids": "article_id_sequence"})
    .with_columns(
        pl.lit("mind_large_test").alias("dataset"),
        pl.lit(None, dtype=pl.List(pl.Datetime)).alias("timestamp_sequence"),
        pl.lit(None, dtype=pl.List(pl.Float64)).alias("read_time_sequence"),
        pl.lit(None, dtype=pl.List(pl.Float64)).alias("scroll_percentage_sequence"),
    )
    .select("user_id", "dataset", "article_id_sequence", "timestamp_sequence", "read_time_sequence", "scroll_percentage_sequence")
)
history_table.write_parquet(OUT_DIR / "history.parquet")

# article_ids_clicked genuinely does not exist for this blind population --
# left out entirely rather than null-filled, see the markdown above.
behaviors_final = behaviors.select(
    "impression_id", "dataset", "user_id", "impression_time", "article_ids_inview", "session_id", "split"
)
behaviors_final.write_parquet(OUT_DIR / "behaviors.parquet")

log_progress(
    f"mind_large_test: parsed {behaviors_final.height} impressions, "
    f"{history_table.height} distinct users, from behaviors.tsv"
)
{"impressions": behaviors_final.height, "distinct_users": history_table.height}

{'impressions': 2370727, 'distinct_users': 702005}

In [11]:
def test_behaviors_parsed():
    assert behaviors_final.height == behaviors_raw.height
    assert behaviors_final["impression_id"].n_unique() == behaviors_final.height
    inview_lens = behaviors_final["article_ids_inview"].list.len()
    assert (inview_lens > 0).all(), "every impression must have at least one candidate"
    assert (behaviors_final["dataset"] == "mind_large_test").all()
    assert (behaviors_final["split"] == "test").all()
    assert behaviors_final["session_id"].is_null().all()
    assert behaviors_final["impression_time"].dtype == pl.Datetime

    # schema parity with mind_large's own unified tables, minus the one
    # column that genuinely cannot exist here (see the markdown above)
    existing_behaviors_schema = pl.read_parquet_schema(ROOT / "data" / "processed" / "mind_large" / "behaviors.parquet")
    assert set(behaviors_final.columns) == set(existing_behaviors_schema.keys()) - {"article_ids_clicked"}
    existing_history_schema = pl.read_parquet_schema(ROOT / "data" / "processed" / "mind_large" / "history.parquet")
    assert set(history_table.columns) == set(existing_history_schema.keys())

    # structural invariant (same one build_pipeline.ipynb's MIND parsing relies
    # on): every user must have exactly one distinct history string across all
    # of their impression rows -- proves history is a fixed pre-window
    # snapshot, not something that could vary per impression.
    check = behaviors_raw.group_by("user_id").agg(pl.col("history").fill_null("").n_unique().alias("n_unique"))
    violations = check.filter(pl.col("n_unique") > 1)
    assert violations.height == 0, f"{violations.height} users have inconsistent history across impressions"


test_behaviors_parsed()
print(
    f"ok: {behaviors_final.height} impressions parsed, unique impression_id, every user has one fixed history, "
    "schema matches mind_large (minus article_ids_clicked)"
)

ok: 2370727 impressions parsed, unique impression_id, every user has one fixed history, schema matches mind_large (minus article_ids_clicked)


## score_inview adapters

Same shape as Q4/Q5's adapters (`score_inview(user_id, article_ids_inview) ->
dict[article_id, float]`), just a single dataset instead of one per
`(dataset)` in `make_score_inview_adapters`. BM25 memoizes the last user's
full-corpus score vector; `cosine_similarity_subset` takes the precomputed
`corpus_unit`/`id_to_idx` from the previous cell (SPEC.md Q4 #9's fix, built
in from the start here rather than needing to be discovered and retrofitted).

In [12]:
history_lookup = dict(zip(history_table["user_id"].to_list(), history_table["article_id_sequence"].to_list()))


def build_user_query_tokens(article_id_sequence, title_lookup: dict, recent_n: int = RECENT_N_CLICKS) -> list[str]:
    recent_ids = list(article_id_sequence)[-recent_n:]
    titles = [title_lookup.get(aid, "") for aid in recent_ids]
    return tokenize(" ".join(t for t in titles if t))


def build_user_query_vector(article_id_sequence, embedding_lookup: dict, recent_n: int = RECENT_N_CLICKS):
    recent_ids = list(article_id_sequence)[-recent_n:]
    return mean_pool(recent_ids, embedding_lookup)


bm25_cache = {"user_id": None, "scores": None}


def bm25_fn(user_id, article_ids_inview):
    if bm25_cache["user_id"] != user_id:
        seq = history_lookup.get(user_id, [])
        query_tokens = build_user_query_tokens(seq, title_by_id)
        bm25_cache["user_id"] = user_id
        bm25_cache["scores"] = get_scores(bm25_index, query_tokens)
    scores = bm25_cache["scores"]
    return {aid: float(scores[id_to_idx[aid]]) for aid in article_ids_inview}


def embedding_fn(user_id, article_ids_inview):
    seq = history_lookup.get(user_id, [])
    query_vector = build_user_query_vector(seq, emb_lookup)
    scored = cosine_similarity_subset(query_vector, corpus_unit, doc_ids, id_to_idx, article_ids_inview)
    return {aid: scored.get(aid, 0.0) for aid in article_ids_inview}


score_inview_adapters = {"bm25": bm25_fn, "embedding": embedding_fn}

In [13]:
def test_score_inview_adapters():
    sample = behaviors_final.row(0, named=True)
    inview = list(sample["article_ids_inview"])
    for method in METHODS:
        scored = score_inview_adapters[method](sample["user_id"], inview)
        assert set(scored) == set(inview)
        assert all(np.isfinite(v) for v in scored.values())

    # a user with no history (cold-start) -> all-zero (tied) scores, not a crash
    coldstart_rows = history_table.filter(pl.col("article_id_sequence").list.len() == 0)
    if coldstart_rows.height > 0:
        coldstart_user = coldstart_rows.row(0, named=True)["user_id"]
        bm25_scored = score_inview_adapters["bm25"](coldstart_user, inview)
        assert set(bm25_scored.values()) == {0.0}
        emb_scored = score_inview_adapters["embedding"](coldstart_user, inview)
        assert set(emb_scored.values()) == {0.0}

    fn = score_inview_adapters["bm25"]
    assert fn(sample["user_id"], inview) == fn(sample["user_id"], inview)


test_score_inview_adapters()
print("ok: score_inview adapters have a uniform signature, no NaN/inf, and score cold-start users as an all-zero tie")

ok: score_inview adapters have a uniform signature, no NaN/inf, and score cold-start users as an all-zero tie


## Generate predictions

Per SPEC.md Q5 #2/#4, same format as `generate_predictions.ipynb`: one line
per impression, `{impression_id} [{rank_1},...,{rank_n}]`, `rank_i` the
1-indexed rank of the *i*-th candidate in `article_ids_inview`'s original
order, row order matching `behaviors.tsv`'s own row order. Computed in a
`user_id`-sorted order for the BM25 adapter's cache, written back out keyed
by `row_idx` -- same trick as `generate_predictions.ipynb`. Zip contains
exactly `prediction.txt` at the root (MIND's confirmed singular filename).

In [14]:
def generate_predictions(method: str) -> Path:
    split_behaviors = behaviors_final.with_row_index("row_idx")
    score_fn = score_inview_adapters[method]

    compute_order = split_behaviors.sort("user_id")
    row_idx_col = compute_order["row_idx"].to_list()
    user_id_col = compute_order["user_id"].to_list()
    inview_col = compute_order["article_ids_inview"].to_list()
    n_rows = len(row_idx_col)
    log_progress(f"mind_large_test/{method}: scoring {n_rows} impressions")

    ranks_by_row = {}
    for i, (row_idx, user_id, inview) in enumerate(zip(row_idx_col, user_id_col, inview_col)):
        inview_ids = list(inview)
        scored = score_fn(user_id, inview_ids)
        scores = np.array([scored[aid] for aid in inview_ids])
        ranks = (np.argsort(np.argsort(-scores, kind="stable"), kind="stable") + 1).tolist()
        ranks_by_row[row_idx] = ranks
        if (i + 1) % 200_000 == 0:
            log_progress(f"    mind_large_test/{method}: {i + 1}/{n_rows} impressions scored")

    lines = []
    for row_idx, impression_id in zip(split_behaviors["row_idx"].to_list(), split_behaviors["impression_id"].to_list()):
        ranks_str = "[" + ",".join(str(r) for r in ranks_by_row[row_idx]) + "]"
        lines.append(f"{impression_id} {ranks_str}")

    txt_path = SUBMISSIONS_DIR / "prediction.txt"
    # write_text() would translate \n -> \r\n on Windows; write bytes
    # directly so the file matches MIND's reference implementation exactly.
    txt_path.write_bytes(("\n".join(lines) + "\n").encode("utf-8"))

    zip_path = SUBMISSIONS_DIR / f"mind_large_test_{method}_predictions.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(txt_path, arcname="prediction.txt")
    txt_path.unlink()
    log_progress(f"mind_large_test/{method}: wrote {zip_path.name} ({len(lines)} lines)")
    return zip_path


prediction_zips = {method: generate_predictions(method) for method in METHODS}
log_progress("mind_large_test_submission: all zips written")
prediction_zips

{'embedding': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/submissions/mind_large_test/mind_large_test_embedding_predictions.zip'),
 'bm25': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/submissions/mind_large_test/mind_large_test_bm25_predictions.zip')}

In [15]:
def test_generate_predictions():
    n_expected = behaviors_final.height
    expected_ids = behaviors_final["impression_id"].to_list()
    expected_inview_lens = [len(x) for x in behaviors_final["article_ids_inview"].to_list()]

    for method in METHODS:
        zip_path = prediction_zips[method]
        assert zip_path.exists()
        with zipfile.ZipFile(zip_path) as zf:
            names = zf.namelist()
            assert names == ["prediction.txt"], f"zip must contain exactly prediction.txt at root, got {names}"
            content = zf.read("prediction.txt").decode("utf-8")

        lines = content.strip("\n").split("\n")
        assert len(lines) == n_expected

        for line, expected_id, expected_len in zip(lines, expected_ids, expected_inview_lens):
            impr_id_str, ranks_str = line.split(" ", 1)
            assert impr_id_str == expected_id, "row order must match behaviors.tsv's original row order"
            ranks = [int(r) for r in ranks_str.strip("[]").split(",")]
            assert len(ranks) == expected_len
            assert sorted(ranks) == list(range(1, expected_len + 1)), "ranks must be a permutation starting at 1"


test_generate_predictions()
log_progress("mind_large_test_submission completed successfully")
print("ok: prediction.txt round-trips for every method -- correct row count, row order, and valid rank permutations")

ok: prediction.txt round-trips for every method -- correct row count, row order, and valid rank permutations


# Manual Review Complete

Unlike `mind_large`'s submission zips (see `generate_predictions.ipynb`'s
intro / SPEC.md Q5 #3), `submissions/mind_large_test/mind_large_test_embedding_predictions.zip`
**is** the real, submittable Codabench population for
`codabench.org/competitions/13967` -- rename `prediction.txt` isn't
needed (already the confirmed filename), just upload the zip directly
under Participate → Submit / View Results. Prefer the `embedding` method
zip for the actual submission (Q4 found embeddings ranking better than
BM25 on every dataset's test split); the `bm25` zip is for comparison only.